In [ ]:
import psycopg2

In [ ]:
DATABASE_URL = "dbname=citus user=citus password=citus host=localhost port=5439"
TABLE_PREFIX = "v3"
PATCH_TABLE = f"{TABLE_PREFIX}_patch"
PRED_PATCH_TABLE = f"{TABLE_PREFIX}_pred_patch"

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Drop tables if they exist
cur.execute(f"DROP TABLE IF EXISTS {PATCH_TABLE} CASCADE;")
cur.execute(f"DROP TABLE IF EXISTS {PRED_PATCH_TABLE} CASCADE;")

conn.commit()
cur.close()
conn.close()

print(f"Tables {PATCH_TABLE} and {PRED_PATCH_TABLE} removed if they existed.")

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

create_table_query = f"""
CREATE UNLOGGED TABLE {PATCH_TABLE}(
    id SERIAL NOT NULL,
    patch_uid integer NOT NULL,
    gt_label integer,
    event_ts timestamp with time zone NOT NULL DEFAULT now(),
    image_id integer,
    working_mag double precision,
    PRIMARY KEY(id)
);
"""

cur.execute(create_table_query)
cur.execute(f"SELECT create_distributed_table('{PATCH_TABLE}', 'id');")

conn.commit()
cur.close()
conn.close()

print(f"Created and distributed {PATCH_TABLE} on 'id'.")

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

create_pred_patch_table_query = f"""
CREATE UNLOGGED TABLE {PRED_PATCH_TABLE}(
    id SERIAL NOT NULL,
    patch_uid bigint NOT NULL,
    embed_coords point,
    grid_cell_i int,
    grid_cell_j int,
    event_ts timestamp with time zone NOT NULL DEFAULT now(),
    pred_label integer,
    patch_coords point,
    PRIMARY KEY(id)
);

CREATE INDEX idx_{PRED_PATCH_TABLE}_grid_cells ON {PRED_PATCH_TABLE} (grid_cell_i, grid_cell_j);
"""

cur.execute(create_pred_patch_table_query)
cur.execute(f"SELECT create_distributed_table('{PRED_PATCH_TABLE}', 'id');")

conn.commit()
cur.close()
conn.close()

print(f"Created and distributed {PRED_PATCH_TABLE} on 'id'.")

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

# Finest level is 12 (4096x4096), coarsening up to level 8 (256x256)
# Divisor = 2^(finest_level - target_level)
levels = [
    ("l8", 16),
    ("l9",  8),   # level 9:  512x512   (divide l12 cells by 8)
    ("l10", 4),   # level 10: 1024x1024 (divide l12 cells by 4)
    ("l11", 2),   # level 11: 2048x2048 (divide l12 cells by 2)
    ("l12", 1),   # level 12: 4096x4096 (finest, no division)
]

In [ ]:
# JOIN on id — both tables are distributed on id, so the join is co-located
statements = []
for level_name, divisor in levels:
    view_name = f"{TABLE_PREFIX}_patch_label_agg_{level_name}"
    index_name = f"idx_{TABLE_PREFIX}_patch_label_agg_{level_name}_grid"
    if divisor > 1:
        i_expr = f"(pp.grid_cell_i / {divisor})"
        j_expr = f"(pp.grid_cell_j / {divisor})"
    else:
        i_expr = "pp.grid_cell_i"
        j_expr = "pp.grid_cell_j"

    statements.append(f"""
CREATE MATERIALIZED VIEW {view_name} AS
SELECT
    pp.pred_label,
    p.gt_label,
    {i_expr} AS grid_cell_i,
    {j_expr} AS grid_cell_j,
    COUNT(*) AS patch_count
FROM {PRED_PATCH_TABLE} pp
LEFT JOIN {PATCH_TABLE} p ON pp.id = p.id
GROUP BY pp.pred_label, p.gt_label, {i_expr}, {j_expr};
""")
    statements.append(f"CREATE INDEX {index_name} ON {view_name} (grid_cell_i, grid_cell_j);")

for stmt in statements:
    cur.execute(stmt)

conn.commit()
cur.close()
conn.close()

print(f"Tables created: {PATCH_TABLE}, {PRED_PATCH_TABLE}")
print(f"Materialized views created: {TABLE_PREFIX}_patch_label_agg_l8 through {TABLE_PREFIX}_patch_label_agg_l12")

In [ ]:
import numpy as np
import psycopg2
import time
from io import StringIO
from multiprocessing import Pool, cpu_count

# ======================
# CONFIG
# ======================
FINEST_LEVEL = 12
GRID_SIZE = 2 ** FINEST_LEVEL
NUM_LABELS = 5
BILLION = 100_000_000
BATCH_SIZE = 500_000
SPATIAL_CLUSTERS = 10
POINTS_PER_CLUSTER = (BILLION - 1_000) // SPATIAL_CLUSTERS
UNIFORM_POINTS = 1_000
# Citus fans out each COPY to all worker nodes, so keep Python-level
# parallelism low to avoid exceeding worker max_connections.
WORKERS = 2

rng = np.random.default_rng(2026)

# ======================
# GRID FUNCTION (vectorized)
# ======================
def vectorized_point_to_cell(embed_x, embed_y, cell_size, level):
    scaled_size = cell_size / (2 ** level)
    i_arr = np.floor(embed_x / scaled_size).astype(np.int32)
    j_arr = np.floor(embed_y / scaled_size).astype(np.int32)
    return i_arr, j_arr


# ======================
# COPY HELPERS
# ======================
def build_patch_buffer(patch_uids, gt_label):
    buf = StringIO()
    for uid in patch_uids:
        buf.write(f"{uid}\t{gt_label}\t\\N\t\\N\n")
    buf.seek(0)
    return buf


def build_pred_buffer(patch_uids, ex, ey, grid_i, grid_j, pred_labels, px, py):
    buf = StringIO()
    for i in range(len(patch_uids)):
        buf.write(
            f"{patch_uids[i]}\t({ex[i]},{ey[i]})\t"
            f"{grid_i[i]}\t{grid_j[i]}\t{pred_labels[i]}\t({px[i]},{py[i]})\n"
        )
    buf.seek(0)
    return buf


def copy_batch(conn, patch_buf, pred_buf, n_rows):
    cur = conn.cursor()

    cur.copy_from(
        patch_buf,
        PATCH_TABLE,
        columns=("patch_uid", "gt_label", "image_id", "working_mag"),
    )

    cur.copy_from(
        pred_buf,
        PRED_PATCH_TABLE,
        columns=(
            "patch_uid",
            "embed_coords",
            "grid_cell_i",
            "grid_cell_j",
            "pred_label",
            "patch_coords",
        ),
    )

    conn.commit()
    cur.close()


# ======================
# WORKER
# ======================
def worker(worker_id):
    conn = psycopg2.connect(DATABASE_URL)
    local_rng = np.random.default_rng(2026 + worker_id)

    uid_start = worker_id * (BILLION // WORKERS) + 1
    uid = uid_start

    t0 = time.perf_counter()

    try:
        clusters_per_worker = SPATIAL_CLUSTERS // WORKERS

        for cluster in range(worker_id * clusters_per_worker,
                             (worker_id + 1) * clusters_per_worker):

            mean_x = local_rng.uniform(GRID_SIZE * 0.1, GRID_SIZE * 0.9)
            mean_y = local_rng.uniform(GRID_SIZE * 0.1, GRID_SIZE * 0.9)
            std = local_rng.uniform(GRID_SIZE * 0.05, GRID_SIZE * 0.2)

            gt_label = cluster % NUM_LABELS
            pred_mode = (
                int(local_rng.integers(0, NUM_LABELS))
                if cluster % 2 == 0 else "random"
            )

            for start in range(0, POINTS_PER_CLUSTER, BATCH_SIZE):
                n = min(BATCH_SIZE, POINTS_PER_CLUSTER - start)

                ex = np.clip(local_rng.normal(mean_x, std, n), 0, GRID_SIZE)
                ey = np.clip(local_rng.normal(mean_y, std, n), 0, GRID_SIZE)

                patch_uids = np.arange(uid, uid + n, dtype=np.int64)

                grid_i, grid_j = vectorized_point_to_cell(
                    ex, ey, GRID_SIZE, FINEST_LEVEL
                )

                pred_labels = (
                    local_rng.integers(0, NUM_LABELS, n, dtype=np.int32)
                    if pred_mode == "random"
                    else np.full(n, pred_mode, dtype=np.int32)
                )

                px = local_rng.uniform(0, 10_000, n)
                py = local_rng.uniform(0, 10_000, n)

                patch_buf = build_patch_buffer(patch_uids, gt_label)
                pred_buf = build_pred_buffer(
                    patch_uids, ex, ey, grid_i, grid_j, pred_labels, px, py
                )

                copy_batch(conn, patch_buf, pred_buf, n)

                uid += n

                if uid % (10_000_000) < n:
                    elapsed = time.perf_counter() - t0
                    print(
                        f"[Worker {worker_id}] {uid - uid_start:,} rows "
                        f"({(uid - uid_start)/elapsed:,.0f} rows/sec)"
                    )

        # Uniform distribution (only one worker handles it)
        if worker_id == 0:
            ex = local_rng.uniform(0, GRID_SIZE, UNIFORM_POINTS)
            ey = local_rng.uniform(0, GRID_SIZE, UNIFORM_POINTS)

            patch_uids = np.arange(uid, uid + UNIFORM_POINTS)

            grid_i, grid_j = vectorized_point_to_cell(
                ex, ey, GRID_SIZE, FINEST_LEVEL
            )

            pred_labels = local_rng.integers(0, NUM_LABELS, UNIFORM_POINTS)

            px = local_rng.uniform(0, 10_000, UNIFORM_POINTS)
            py = local_rng.uniform(0, 10_000, UNIFORM_POINTS)

            patch_buf = build_patch_buffer(patch_uids, NUM_LABELS)
            pred_buf = build_pred_buffer(
                patch_uids, ex, ey, grid_i, grid_j, pred_labels, px, py
            )

            copy_batch(conn, patch_buf, pred_buf, UNIFORM_POINTS)

    finally:
        conn.close()

    return worker_id


# ======================
# MAIN
# ======================
def run():
    t0 = time.perf_counter()

    with Pool(WORKERS) as p:
        p.map(worker, range(WORKERS))

    elapsed = time.perf_counter() - t0
    print(f"\nInserted ~{BILLION:,} rows in {elapsed:.2f}s")
    print(f"Throughput: {BILLION/elapsed:,.0f} rows/sec")


if __name__ == "__main__":
    run()

In [ ]:
import time

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

timings = {}
for level_name, _ in levels:
    view_name = f"{TABLE_PREFIX}_patch_label_agg_{level_name}"
    t0 = time.perf_counter()
    cur.execute(f"REFRESH MATERIALIZED VIEW {view_name};")
    t1 = time.perf_counter()
    timings[view_name] = t1 - t0

conn.commit()
cur.close()
conn.close()

for view, t in timings.items():
    print(f"Refreshed {view} in {t:.3f} seconds.")

In [ ]:
conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

query = f"SELECT * FROM {view_name} LIMIT 10;"
cur.execute(query)

rows = cur.fetchall()
for row in rows:
    print(row)

cur.close()
conn.close()

## 10 – Query 4: Max patch count per (gt_label, pred_label) pair

Benchmark the time to execute a GROUP BY aggregation returning the maximum
`patch_count` for every ground-truth / prediction label combination.

In [ ]:
import time

conn = psycopg2.connect(DATABASE_URL)
cur = conn.cursor()

view_name = f"{TABLE_PREFIX}_patch_label_agg_l8"
query = f"""
    SELECT gt_label, pred_label, MAX(patch_count) AS max_count
    FROM {view_name}
    GROUP BY gt_label, pred_label
    ORDER BY gt_label, pred_label;
"""

N_RUNS = 10
times = []
for _ in range(N_RUNS):
    t0 = time.perf_counter()
    cur.execute(query)
    rows = cur.fetchall()
    t1 = time.perf_counter()
    times.append(t1 - t0)

cur.close()
conn.close()

times_ms = [t * 1000 for t in times]
print(f"Rows returned: {len(rows)}")
print(f"Runs: {N_RUNS}")
print(f"Min:    {min(times_ms):.2f} ms")
print(f"Max:    {max(times_ms):.2f} ms")
print(f"Mean:   {sum(times_ms)/len(times_ms):.2f} ms")
print(f"Median: {sorted(times_ms)[N_RUNS//2]:.2f} ms")
print("\nSample results (first 10 rows):")
for row in rows[:10]:
    print(f"  gt={row[0]}, pred={row[1]}, max_count={row[2]}")